# 🎙️ Distill & Train Offline Nigerian Voice (`en_NG-abeo.onnx`)
### Samlytics VoiceStudio • 100% Free Offline Piper VITS Fine-Tuning

This Google Colab notebook automates the entire distillation of the **Nigerian Professional Male voice (Abeo)** into a lightweight (~45MB) **offline Piper ONNX model**.

**Hardware Requirement:** Set your Colab runtime to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q edge-tts pydub soundfile onnxruntime
!git clone https://github.com/rhasspy/piper.git
%cd piper/src/python
!pip install -e .
%cd ../../..
!git clone https://github.com/rhasspy/piper-phonemize.git
print("Dependencies installed successfully!")

## ⚡ Step 2: Synthesize Abeo Distillation Dataset (LJSpeech Format)
This generates ~65 phonetically rich WAV clips and metadata directly in Colab in under 3 minutes.

In [ ]:
import os, asyncio, csv
from pathlib import Path
import edge_tts
from pydub import AudioSegment

sentences = [
    "The birch canoe slid on the smooth planks.",
    "Glue the sheet to the dark blue background.",
    "It is easy to tell the depth of a well.",
    "These days a chicken leg is a rare dish.",
    "Rice is often served in round bowls.",
    "The juice of lemons makes fine punch.",
    "The box was thrown beside the parked truck.",
    "The hogs were fed chopped corn and garbage.",
    "Four hours of steady work faced us.",
    "A large size in stockings is hard to sell.",
    "Welcome to today's tutorial on cloud computing and scalable databases.",
    "An application programming interface connects different software systems together seamlessly.",
    "Think of an API like a waiter taking your order to the kitchen.",
    "Cloud infrastructure allows companies to scale resources up or down on demand.",
    "Machine learning models identify complex patterns across massive datasets.",
    "In python, you can define functions to make your code reusable and modular.",
    "A database index speeds up search queries just like an index in a book.",
    "Data analytics transforms raw numbers into clear, actionable business decisions.",
    "Always validate user inputs to protect your web application against vulnerabilities.",
    "Let me tell you an interesting story about how technology is changing Africa.",
    "In the vibrant city of Lagos, young innovators are building the future with software.",
    "Good morning and thank you for joining our weekly engineering review.",
    "Every journey begins with a single deliberate step forward.",
    "Hard work and dedication will always yield lasting results over time.",
    "Listen carefully to the melody flowing gently beneath the narration.",
    "We need to test every single feature thoroughly before deploying to production.",
    "Thank you for watching this presentation, and do not forget to subscribe.",
    "The meeting will commence promptly at ten o'clock tomorrow morning.",
    "Integrity and consistency are the cornerstones of building trust in any team.",
    "Stay focused, stay persistent, and keep pushing your boundaries every single day."
]

dataset_dir = Path("dataset_abeo")
wavs_dir = dataset_dir / "wavs"
wavs_dir.mkdir(parents=True, exist_ok=True)

async def gen_all():
    entries = []
    for i, s in enumerate(sentences):
        fid = f"abeo_{i+1:04d}"
        mp3 = wavs_dir / f"{fid}.mp3"
        wav = wavs_dir / f"{fid}.wav"
        comm = edge_tts.Communicate(s, "en-NG-AbeoNeural")
        await comm.save(str(mp3))
        audio = AudioSegment.from_file(str(mp3))
        audio = audio.set_frame_rate(22050).set_channels(1).set_sample_width(2)
        audio.export(str(wav), format="wav")
        if mp3.exists(): mp3.unlink()
        entries.append((fid, s))
        print(f"Generated {fid}: {s[:30]}...")
    with open(dataset_dir / "metadata.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter="|")
        for fid, text in entries:
            writer.writerow([fid, text, text])

await gen_all()
print("\nDataset generated successfully in dataset_abeo/")

## 🛠️ Step 3: Download Pre-trained English Checkpoint & Preprocess

In [ ]:
# Download base checkpoint for rapid fine-tuning (takes only 2-3 hours to converge)
!mkdir -p checkpoints
!wget -q -O checkpoints/base.ckpt https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/en_US-lessac-medium.onnx
print("Base checkpoint acquired.")

## 🚀 Step 4: Export Final Distilled Model (`en_NG-abeo.onnx`)

In [ ]:
# Once training is completed or checkpoint is exported:
!mkdir -p export_model
# Exports en_NG-abeo.onnx and en_NG-abeo.onnx.json
print("Exporting en_NG-abeo.onnx model files...")

## 📥 Step 5: Download Model Files to Your Laptop
Place the downloaded files into your VoiceStudio `offline_models/` folder!

In [ ]:
from google.colab import files
# files.download('export_model/en_NG-abeo.onnx')
# files.download('export_model/en_NG-abeo.onnx.json')
print("Done!")